In [1]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [2]:
# Lab results from hosp.labevents for cohort patients.
# labevents has no ed_stay_id — joined via subject_id + time window.
# Window covers:
#   - ED-only patients: ed_intime to ed_outtime
#   - Admitted patients: ed_intime to first_icu_intime (if ICU transfer occurred) or dischtime
# Labs ordered after ICU transfer are excluded from the window entirely.
# order_time = charttime (when the order was placed / specimen collected)
# result_time = storetime (when the result was filed)
# ordered_location: 'ed' if order placed during ED window, 'ward' after ED discharge
# result_after_icu_transfer: True if result_time >= first_icu_intime — lab was ordered before
#   ICU transfer but result came back after (retrospective label, not a state feature)
# hadm_id is NULL for ED-only patients, populated for admitted patients.
# NOTE: This query may take several minutes due to labevents table size.

query_labs = """
WITH cohort_subjects AS (
  SELECT
    ed_stay_id,
    subject_id,
    hadm_id,
    ed_intime,
    ed_outtime,
    first_icu_intime,
    -- window_end caps at ICU transfer for admitted patients who went to ICU
    CASE
      WHEN hadm_id IS NOT NULL AND first_icu_intime IS NOT NULL THEN first_icu_intime
      WHEN hadm_id IS NOT NULL THEN dischtime
      ELSE ed_outtime
    END AS window_end
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  cs.ed_stay_id,
  le.subject_id,
  cs.hadm_id,
  le.labevent_id,
  le.itemid,
  dl.label,
  dl.fluid,
  dl.category,
  le.charttime AS order_time,
  le.storetime AS result_time,
  le.flag,
  CASE
    WHEN le.charttime <= cs.ed_outtime THEN 'ed'
    ELSE 'ward'
  END AS ordered_location,
  CASE
    WHEN cs.first_icu_intime IS NOT NULL AND le.storetime >= cs.first_icu_intime THEN TRUE
    ELSE FALSE
  END AS result_after_icu_transfer
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN cohort_subjects cs
  ON le.subject_id = cs.subject_id
  AND le.charttime >= cs.ed_intime
  AND le.charttime < cs.window_end
INNER JOIN `physionet-data.mimiciv_3_1_hosp.d_labitems` dl
  ON le.itemid = dl.itemid
ORDER BY cs.ed_stay_id, le.charttime
"""

print("Running labs query (may take several minutes)...")
df_labs = client.query(query_labs).to_dataframe()
print(f"Shape: {df_labs.shape}")
print(f"Unique ED visits with labs: {df_labs['ed_stay_id'].nunique():,}")
print(f"\nOrdered location breakdown:\n{df_labs['ordered_location'].value_counts()}")
print(f"\nResult after ICU transfer: {df_labs['result_after_icu_transfer'].sum():,} rows")
print(f"\nCategory x fluid combinations: {df_labs.groupby(['category', 'fluid']).ngroups}")
df_labs.head()

Running labs query (may take several minutes)...
Shape: (31072620, 13)
Unique ED visits with labs: 312,217

Ordered location breakdown:
ordered_location
ward    15561929
ed      15510691
Name: count, dtype: int64

Result after ICU transfer: 152,506 rows

Category x fluid combinations: 19


,ed_stay_id,subject_id,hadm_id,labevent_id,itemid,label,fluid,category,order_time,result_time,flag,ordered_location,result_after_icu_transfer
0,30000012,11714491,21562392,26869933,50882,Bicarbonate,Blood,Chemistry,2126-02-14 22:15:00,2126-02-14 23:04:00,NaN,ed,False
1,30000012,11714491,21562392,26869942,51133,Absolute Lymphocyte Count,Blood,Hematology,2126-02-14 22:15:00,2126-02-14 22:36:00,NaN,ed,False
2,30000012,11714491,21562392,26869958,52073,Absolute Eosinophil Count,Blood,Hematology,2126-02-14 22:15:00,2126-02-14 22:36:00,NaN,ed,False
3,30000012,11714491,21562392,26869932,50878,Asparate Aminotransferase (AST),Blood,Chemistry,2126-02-14 22:15:00,2126-02-14 23:04:00,NaN,ed,False
4,30000012,11714491,21562392,26869961,52135,Immature Granulocytes,Blood,Hematology,2126-02-14 22:15:00,2126-02-14 22:36:00,NaN,ed,False


In [ ]:
PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

info = DatasetInfo(
    description=(
        "Laboratory results from hosp.labevents for cohort patients during their stay window. "
        "Grouped by category x fluid (19 unique combinations) rather than individual test label "
        "to reduce action space. Each result includes the abnormal flag for worst-case aggregation. "
        "Intended as the primary lab state feature source."
    ),
    license=PHYSIONET_LICENSE,
)

ds = Dataset.from_pandas(df_labs, split='labs_base', info=info)
del df_labs # Do this for memory issues

ds.push_to_hub("ADS599-Capstone/raw_data", config_name="labs_base", data_dir="lab_events")
print("Pushed labs_base to HuggingFace Hub.")